# Build the fine-tuned DeFiScope **Phi-3** model → portable GGUF for Ollama

This notebook turns the authors' released LoRA adapter into a single **~8 GB GGUF**
file you can run locally with Ollama (on a Mac GPU via Metal, etc.).

**What it does:** downloads `microsoft/Phi-3-medium-128k-instruct` (the base), merges
the released LoRA adapter `RocketRaccoonnn/...lora_v2` into it, converts to GGUF, and
quantizes to Q4_K_M.

### Runtime you need (this step, ONCE — not your laptop)
- **Best:** an **A100 40 GB** GPU runtime (Colab Pro, or a cloud box ~$0.50 for the job) — merge is fast and reliable.
- **Free option:** a **high-RAM CPU** runtime (Kaggle ~30 GB RAM, or Colab Pro High-RAM 51 GB). The merge runs on CPU; slower but works.
- **Disk:** ~70 GB free (Colab/Kaggle have enough; the notebook deletes intermediates as it goes).
- ⚠️ Free Colab base tier (T4 16 GB + ~12 GB RAM) is **too small** to merge a 14B model — use one of the above.

You run this once. The output GGUF then runs on your 16 GB laptop forever.


## 1. Install dependencies


In [ ]:
!pip -q install "transformers>=4.44" "peft>=0.11" "accelerate>=0.30" sentencepiece huggingface_hub
import torch, transformers, peft
print('torch', torch.__version__, '| transformers', transformers.__version__, '| CUDA', torch.cuda.is_available())


## 2. Merge the LoRA adapter into the base model

Loads the base in fp16 and folds the 7.4 MB adapter in with `merge_and_unload()`.
Auto-uses the GPU if it has >30 GB VRAM, otherwise CPU (needs ~28 GB system RAM).


In [ ]:
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE    = 'microsoft/Phi-3-medium-128k-instruct'
ADAPTER = 'RocketRaccoonnn/Phi-3-medium-128k-instruct_LoRA_CASUAL_LM_lora_v2'
MERGED  = 'defiscope-phi3-merged'

# Pick device: a big GPU (>30GB) can hold the whole fp16 model; else load on CPU.
big_gpu = torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory > 30e9
device_map = 'cuda' if big_gpu else 'cpu'
print('Loading base on:', device_map, '(this downloads ~28 GB and takes a few minutes)')

tok  = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
base = AutoModelForCausalLM.from_pretrained(
    BASE, torch_dtype=torch.float16, low_cpu_mem_usage=True,
    trust_remote_code=True, device_map=device_map)

print('Applying + merging LoRA adapter...')
model = PeftModel.from_pretrained(base, ADAPTER)
model = model.merge_and_unload()          # fold LoRA into the base weights

model.save_pretrained(MERGED, safe_serialization=True)
tok.save_pretrained(MERGED)
print('Merged model saved to', MERGED)

del model, base; gc.collect()
torch.cuda.is_available() and torch.cuda.empty_cache()
# free the HF download cache (~28 GB) to make room for GGUF conversion
!rm -rf /root/.cache/huggingface /root/.cache/huggingface/hub ~/.cache/huggingface


## 3. Convert to GGUF and quantize to Q4_K_M

Uses llama.cpp. Produces `defiscope-phi3-Q4_K_M.gguf` (~8 GB) and deletes the big
intermediate to stay within disk limits.


In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip -q install -r llama.cpp/requirements.txt

# HF model dir -> GGUF (fp16)
!python llama.cpp/convert_hf_to_gguf.py defiscope-phi3-merged \
    --outfile defiscope-phi3-f16.gguf --outtype f16

# free the merged safetensors (~28 GB) now that we have the GGUF
!rm -rf defiscope-phi3-merged

# build the quantizer and quantize to Q4_K_M
!cmake -B llama.cpp/build llama.cpp -DCMAKE_BUILD_TYPE=Release >/dev/null 2>&1
!cmake --build llama.cpp/build --target llama-quantize -j >/dev/null 2>&1
!./llama.cpp/build/bin/llama-quantize defiscope-phi3-f16.gguf defiscope-phi3-Q4_K_M.gguf Q4_K_M

!rm -f defiscope-phi3-f16.gguf
!ls -lh defiscope-phi3-Q4_K_M.gguf


## 4. Save / download the GGUF

An 8 GB direct download can be flaky — saving to Google Drive is more reliable.
Pick one:


In [ ]:
# Option A - Google Drive (recommended for 8 GB)
from google.colab import drive
drive.mount('/content/drive')
!cp defiscope-phi3-Q4_K_M.gguf /content/drive/MyDrive/
print('Copied to your Google Drive. Download it from there to your laptop.')

# Option B - direct browser download (uncomment if you prefer)
# from google.colab import files
# files.download('defiscope-phi3-Q4_K_M.gguf')


## 5. Run it locally (on your laptop, in a normal terminal)

Move `defiscope-phi3-Q4_K_M.gguf` into `DeFiScope/local_phi3/`, then:

```bash
cd DeFiScope/local_phi3
ollama create defiscope-phi3 -f Modelfile      # registers the model with Ollama

cd ..                                          # back to the repo root
source .venv/bin/activate
export OPENAI_API_KEY=ollama                   # any placeholder; Ollama ignores it
export DEFISCOPE_OPENAI_BASE_URL=http://localhost:11434/v1
export DEFISCOPE_OPENAI_MODEL=defiscope-phi3
export ETHERSCAN_API_KEY=YourFreeEtherscanV2Key   # optional, for Type-I source prompts
python main.py -tx 0xca4d0d24aa448329b7d4eb81be653224a59e7b081fc7a1c9aad59c5a38d0ae19 -bp bsc
```

That runs the **fine-tuned DeFiScope Phi-3** entirely on your machine (GPU via Metal),
no OpenAI cost. See `local_phi3/README.md` for details and expectations.
